In [9]:
from dotenv import load_dotenv
import os
import time
import json
import requests
import pandas as pd

# Ruta absoluta o relativa al .env en la raíz del repo
env_path = os.path.abspath(os.path.join(os.getcwd(), "../../..", ".env"))
load_dotenv(dotenv_path=env_path)

# Obtener clave de Grok desde .env
CLAUDE_API_KEY = os.getenv("CLAUDE_API_KEY")

if CLAUDE_API_KEY:
    print("Clave de Claude cargada correctamente.")
else:
    raise ValueError("No se encontró la clave de Claude. Verifica la ruta del .env.")

Clave de Claude cargada correctamente.


In [10]:
def call_claude_api(prompt, text):
    """
    Sends a text and a prompt to Claude Sonnet 3.5 and returns the generated summary.
    """
    url = "https://api.anthropic.com/v1/messages"

    headers = {
        "x-api-key": CLAUDE_API_KEY,
        "anthropic-version": "2023-06-01",
        "content-type": "application/json"
    }

    payload = {
        "model": "claude-sonnet-4-5",
        "max_tokens": 1024,
        "system": "You are an expert assistant specialized in simplifying biomedical language.",
        "messages": [
            {"role": "user", "content": f"{prompt}\n\nText:\n{text}"}
        ]
    }

    try:
        start_time = time.time()
        response = requests.post(url, headers=headers, json=payload)
        elapsed = time.time() - start_time

        print(f"Status code: {response.status_code}")
        if response.status_code == 200:
            data = response.json()
            print("Response keys:", data.keys())
            output = data["content"][0]["text"]
            return output.strip(), elapsed
        else:
            print(f"HTTP error {response.status_code}: {response.text[:200]}")
            return None, elapsed

    except Exception as e:
        print("Request error:", e)
        return None, None

In [11]:
prompt = "Summarize this biomedical paragraph in plain language for the public."
text = "The administration of statins has been shown to reduce LDL cholesterol and cardiovascular risk."
resumen, tiempo = call_claude_api(prompt, text)
print(resumen)
print(f"Time: {tiempo:.2f} s")

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
# Simplified Summary

Statin medications help lower "bad" cholesterol (LDL) in the blood and reduce the risk of heart disease and stroke.

---

**Key terms explained:**
- **Statins**: Common cholesterol-lowering drugs
- **LDL cholesterol**: "Bad" cholesterol that can clog arteries
- **Cardiovascular risk**: Chance of heart attack or stroke
Time: 4.71 s


In [12]:
ruta_dataset = "../data-sources/pre-processed/data_finetuning_test.csv"
df = pd.read_csv(ruta_dataset, encoding="utf-8", on_bad_lines="skip")
display(df.head(2))

,name,article,summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...


In [ ]:
# Prompt para Claude
prompt = """ou are a helpful medical/health writer.
Summarize the following scientific text into a clear summary intended for a general audience.
Do NOT use headings, titles, bullet points, or numbered lists. Do not invent data or references.
If you must use a technical term, briefly define it."""

# Tomar las dos primeras filas del dataset
df_claude_test = df.head(2).copy()

# Lista donde guardaremos los resúmenes generados
gen_summaries = []

# Procesar las dos filas
for i, fila in df_claude_test.iterrows():
    print(f"\n Processing {fila['name']} ({i+1}/{len(df_claude_test)})...\n")
    resumen, tiempo = call_claude_api(prompt, fila['article'])
    gen_summaries.append(resumen if resumen else "")
    print(f"Response time: {tiempo:.2f} s\n")

# Agregar columna con los resúmenes generados
df_claude_test["gen_summary"] = gen_summaries

# Guardar en CSV
ruta_salida = "./prueba_2filasclaude.csv"
df_claude_test.to_csv(ruta_salida, index=False, encoding="utf-8")

print(f"Results saved to: {os.path.abspath(ruta_salida)}")

# Mostrar los resultados en pantalla
display(df_claude_test)


📄 Processing 10.1002-14651858.CD009781.pub2 (1/2)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
⏱️ Response time: 12.17 s


📄 Processing 10.1002-14651858.CD010694.pub2 (2/2)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
⏱️ Response time: 9.63 s

Results saved to: c:\Users\braya\OneDrive\Documentos\GitHub\Proyecto-PLN-FLAG\src\api_tests\prueba_2filasclaude.csv


,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,**Summary**\n\nCorneal abrasions—scratches on ...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,**Summary**\n\nVenous leg ulcers are long-last...


In [14]:
# Código para procesar los 380 registros con Claude
# Prompt para Claude
prompt = """You are a helpful medical/health writer.
Summarize the following scientific text into a clear summary intended for a general audience.
Do NOT use headings, titles, bullet points, or numbered lists. Do not invent data or references.
If you must use a technical term, briefly define it."""

# Copiamos el dataset completo
df_claude_final = df.copy()

# Lista donde guardaremos los resúmenes generados
gen_summaries = []

# Procesar todas las filas
for i, fila in df_claude_final.iterrows():
    print(f"\n Processing {fila['name']} ({i+1}/{len(df_claude_final)})...\n")
    resumen, tiempo = call_claude_api(prompt, fila['article'])
    gen_summaries.append(resumen if resumen else "")
    print(f"⏱️ Response time: {tiempo:.2f} s\n")

# Agregar la columna con los resúmenes generados
df_claude_final["gen_summary"] = gen_summaries

# Guardar en CSV
ruta_salida = "./results_claude.csv"
df_claude_final.to_csv(ruta_salida, index=False, encoding="utf-8")

print(f"Results saved to: {os.path.abspath(ruta_salida)}")

# Mostrar los primeros registros generados
display(df_claude_final.head(3))


 Processing 10.1002-14651858.CD009781.pub2 (1/380)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
⏱️ Response time: 11.73 s


 Processing 10.1002-14651858.CD010694.pub2 (2/380)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
⏱️ Response time: 10.17 s


 Processing 10.1002-14651858.CD009416.pub2 (3/380)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
⏱️ Response time: 10.96 s


 Processing 10.1002-14651858.CD004104.pub4 (4/380)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
⏱️ Response time: 10.90 s


 Processing 10.1002-14651858.CD012689.pub2 (5/380)...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'sto

,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,**Summary for General Audience:**\n\nCorneal a...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,**Summary for General Audience**\n\nVenous leg...
2,10.1002-14651858.CD009416.pub2,Background\r\nThere is currently no strong con...,Which treatments are effective for the treatme...,## Plain Language Summary\n\n**Background**\nC...


In [17]:
# Ruta del archivo csv guardado para verificar su contenido.
ruta_csv = "./results_claude.csv"
df_check = pd.read_csv(ruta_csv)


df_check.info()

df_check.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         380 non-null    object
 1   article      380 non-null    object
 2   summary      380 non-null    object
 3   gen_summary  379 non-null    object
dtypes: object(4)
memory usage: 12.0+ KB


,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,**Summary for General Audience:**\n\nCorneal a...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,**Summary for General Audience**\n\nVenous leg...
2,10.1002-14651858.CD009416.pub2,Background\r\nThere is currently no strong con...,Which treatments are effective for the treatme...,## Plain Language Summary\n\n**Background**\nC...
3,10.1002-14651858.CD004104.pub4,Background\r\nNon‐invasive ventilation (NIV) w...,Non‐invasive ventilation for people with respi...,Summary for General Audience:\n\nWhen people w...
4,10.1002-14651858.CD012689.pub2,Background\r\nSpace spraying is the dispersal ...,Insecticide space spraying for preventing mala...,## Plain Language Summary\n\n**Background**\n\...


In [18]:
# Filtrar la fila faltante
faltante = df_check[df_check["gen_summary"].isnull()]
print(f"🔍 Faltan {len(faltante)} resúmenes.")
display(faltante[["name", "article"]])

🔍 Faltan 1 resúmenes.


,name,article
203,10.1002-14651858.CD004796.pub2,Background\r\nInfantile colic is a common diso...


In [19]:
# Reprocesar solo esa fila
if not faltante.empty:
    for i, fila in faltante.iterrows():
        print(f"\n♻️ Reprocesando {fila['name']}...\n")
        resumen, tiempo = call_claude_api(prompt, fila['article'])
        df_check.loc[i, "gen_summary"] = resumen if resumen else ""
        print(f"✅ Reparado en {tiempo:.2f} s")

# Guardar nuevamente el CSV actualizado
df_check.to_csv(ruta_csv, index=False, encoding="utf-8")
print(f"✅ Archivo actualizado: {os.path.abspath(ruta_csv)}")


♻️ Reprocesando 10.1002-14651858.CD004796.pub2...

Status code: 200
Response keys: dict_keys(['model', 'id', 'type', 'role', 'content', 'stop_reason', 'stop_sequence', 'usage'])
✅ Reparado en 12.79 s
✅ Archivo actualizado: c:\Users\braya\OneDrive\Documentos\GitHub\Proyecto-PLN-FLAG\src\api_tests\results_claude.csv


In [20]:
# Ruta del archivo csv guardado para verificar su contenido.
ruta_csv = "./results_claude.csv"
df_check = pd.read_csv(ruta_csv)


df_check.info()

df_check.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         380 non-null    object
 1   article      380 non-null    object
 2   summary      380 non-null    object
 3   gen_summary  380 non-null    object
dtypes: object(4)
memory usage: 12.0+ KB


,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,**Summary for General Audience:**\n\nCorneal a...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,**Summary for General Audience**\n\nVenous leg...
2,10.1002-14651858.CD009416.pub2,Background\r\nThere is currently no strong con...,Which treatments are effective for the treatme...,## Plain Language Summary\n\n**Background**\nC...
3,10.1002-14651858.CD004104.pub4,Background\r\nNon‐invasive ventilation (NIV) w...,Non‐invasive ventilation for people with respi...,Summary for General Audience:\n\nWhen people w...
4,10.1002-14651858.CD012689.pub2,Background\r\nSpace spraying is the dispersal ...,Insecticide space spraying for preventing mala...,## Plain Language Summary\n\n**Background**\n\...
